In [1]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.*"],
    check=True
)

from pathlib import Path
import random
import shutil
import yaml
import numpy as np
import pandas as pd
import cv2
import zipfile

from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# ============================================================
# 1. EXACT PATHS
# ============================================================

COMPANY_ROOT = Path(
    "/kaggle/input/datasets/tanvirmahtabtaneem/company-bangla-pseudo-training"
)

COMPANY_IMAGES = COMPANY_ROOT / "images"
COMPANY_LABELS = COMPANY_ROOT / "labels"
COMPANY_SUMMARY = COMPANY_ROOT / "pseudo_label_summary.csv"

PUBLIC_ROOT = Path(
    "/kaggle/input/datasets/tanvirmahtabtaneem/"
    "ocr-dataset/Bangla LPDB - A/Bangla_License_Plate"
)

CLASS_FILE = PUBLIC_ROOT / "classes.txt"

CURRENT_MODEL = Path(
    "/kaggle/input/notebooks/tanvirmahtabtaneem/"
    "image-annotation/best.pt"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/company_pseudo_finetune"
)

TRAIN_ROOT = OUTPUT_ROOT / "dataset"
DATA_YAML = TRAIN_ROOT / "data.yaml"
NEW_MODEL = OUTPUT_ROOT / "best.pt"
MODEL_ZIP = Path("/kaggle/working/company_bangla_best.zip")
RESULT_CSV = OUTPUT_ROOT / "model_comparison.csv"



In [3]:
# ============================================================
# 2. SETTINGS
# ============================================================

SEED = 42
IMG_SIZE = 960
BATCH_SIZE = 4
EPOCHS = 30
LR0 = 0.0001
DEVICE = 0

# Use only company images whose pseudo-labels have
# average confidence >= 0.80.
MIN_COMPANY_AVG_CONF = 0.80

random.seed(SEED)
np.random.seed(SEED)

In [4]:
# ============================================================
# 3. CHECK INPUTS
# ============================================================

for path in [
    COMPANY_IMAGES,
    COMPANY_LABELS,
    COMPANY_SUMMARY,
    PUBLIC_ROOT,
    CLASS_FILE,
    CURRENT_MODEL,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing path:\n{path}"
        )

print("=" * 70)
print("INPUTS OK")
print("=" * 70)

print("Current model :", CURRENT_MODEL)
print("Public dataset:", PUBLIC_ROOT)
print("Company images:", COMPANY_IMAGES)
print("Company labels:", COMPANY_LABELS)


INPUTS OK
Current model : /kaggle/input/notebooks/tanvirmahtabtaneem/image-annotation/best.pt
Public dataset: /kaggle/input/datasets/tanvirmahtabtaneem/ocr-dataset/Bangla LPDB - A/Bangla_License_Plate
Company images: /kaggle/input/datasets/tanvirmahtabtaneem/company-bangla-pseudo-training/images
Company labels: /kaggle/input/datasets/tanvirmahtabtaneem/company-bangla-pseudo-training/labels


In [5]:
# ============================================================
# 4. LOAD CLASSES
# ============================================================

class_names = [
    x.strip()
    for x in CLASS_FILE.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines()
    if x.strip()
]

if len(class_names) != 102:
    raise RuntimeError(
        f"Expected 102 classes, found {len(class_names)}"
    )

print("Classes:", len(class_names))

Classes: 102


In [6]:
# ============================================================
# 5. RECREATE ORIGINAL LPDB-A 80/10/10 SPLIT
# ============================================================

public_pairs = []

for image_path in sorted(PUBLIC_ROOT.iterdir()):

    if image_path.suffix.lower() not in [
        ".jpg", ".jpeg", ".png"
    ]:
        continue

    label_path = image_path.with_suffix(".txt")

    if label_path.exists():
        public_pairs.append(
            (image_path, label_path)
        )

random.seed(SEED)
random.shuffle(public_pairs)

total = len(public_pairs)

train_end = int(total * 0.80)
val_end = int(total * 0.90)

public_train = public_pairs[:train_end]
public_val = public_pairs[train_end:val_end]
public_test = public_pairs[val_end:]

print("\nLPDB-A split")
print("Train:", len(public_train))
print("Val  :", len(public_val))
print("Test :", len(public_test))



LPDB-A split
Train: 2134
Val  : 267
Test : 267


In [7]:
# ============================================================
# 6. SELECT COMPANY PSEUDO-LABELED IMAGES
# ============================================================

summary = pd.read_csv(
    COMPANY_SUMMARY
)

summary["image"] = (
    summary["image"].astype(str)
)

company_pairs = []

for image_path in sorted(
    COMPANY_IMAGES.iterdir()
):

    if image_path.suffix.lower() not in [
        ".jpg", ".jpeg", ".png", ".bmp", ".webp"
    ]:
        continue

    label_path = (
        COMPANY_LABELS
        / f"{image_path.stem}.txt"
    )

    if not label_path.exists():
        continue

    if label_path.stat().st_size == 0:
        continue

    rows = summary[
        summary["image"] == image_path.name
    ]

    if rows.empty:
        continue

    avg_conf = float(
        rows.iloc[0]["average_confidence"]
    )

    if avg_conf < MIN_COMPANY_AVG_CONF:
        continue

    company_pairs.append(
        (image_path, label_path)
    )

print(
    "\nCompany pseudo-labels used:",
    len(company_pairs)
)

print(
    "Minimum average confidence:",
    MIN_COMPANY_AVG_CONF
)

if not company_pairs:
    raise RuntimeError(
        "No company pseudo-labels passed "
        "the confidence filter."
    )



Company pseudo-labels used: 1917
Minimum average confidence: 0.8


In [8]:
# ============================================================
# 7. CREATE TRAINING FOLDERS
# ============================================================

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

(
    TRAIN_ROOT
    / "images"
    / "train"
).mkdir(parents=True)

(
    TRAIN_ROOT
    / "images"
    / "val"
).mkdir(parents=True)

(
    TRAIN_ROOT
    / "labels"
    / "train"
).mkdir(parents=True)

(
    TRAIN_ROOT
    / "labels"
    / "val"
).mkdir(parents=True)


def copy_pair(
    image_path,
    label_path,
    split,
    prefix=""
):

    shutil.copy2(
        image_path,
        (
            TRAIN_ROOT
            / "images"
            / split
            / f"{prefix}{image_path.name}"
        )
    )

    shutil.copy2(
        label_path,
        (
            TRAIN_ROOT
            / "labels"
            / split
            / f"{prefix}{label_path.name}"
        )
    )


# Public labeled training data
for image_path, label_path in public_train:

    copy_pair(
        image_path,
        label_path,
        "train"
    )


# Public validation data
for image_path, label_path in public_val:

    copy_pair(
        image_path,
        label_path,
        "val"
    )


# Company pseudo-labeled training data
for i, (image_path, label_path) in enumerate(
    company_pairs
):

    copy_pair(
        image_path,
        label_path,
        "train",
        f"company_{i:05d}_"
    )



In [9]:
# ============================================================
# 8. CREATE data.yaml
# ============================================================

yaml_data = {
    "path": str(TRAIN_ROOT),
    "train": "images/train",
    "val": "images/val",
    "nc": 102,
    "names": class_names
}

DATA_YAML.write_text(
    yaml.safe_dump(
        yaml_data,
        allow_unicode=True,
        sort_keys=False
    ),
    encoding="utf-8"
)

train_count = len(
    list(
        (
            TRAIN_ROOT
            / "images"
            / "train"
        ).glob("*")
    )
)

val_count = len(
    list(
        (
            TRAIN_ROOT
            / "images"
            / "val"
        ).glob("*")
    )
)

print("\nFINAL TRAINING DATA")
print("Public train :", len(public_train))
print("Company train:", len(company_pairs))
print("Total train  :", train_count)
print("Validation   :", val_count)



FINAL TRAINING DATA
Public train : 2134
Company train: 1917
Total train  : 4051
Validation   : 267


In [10]:
# ============================================================
# 9. LOAD CURRENT BEST MODEL
# ============================================================

model = YOLO(
    str(CURRENT_MODEL)
)

print(
    "\nCurrent best.pt loaded."
)



Current best.pt loaded.


In [11]:
# ============================================================
# 10. FINE-TUNE
# ============================================================

print("\n" + "=" * 70)
print("STARTING FINE-TUNING")
print("=" * 70)

model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=2,
    optimizer="AdamW",
    lr0=LR0,
    lrf=0.01,
    weight_decay=0.0005,
    amp=True,
    cache=False,

    mosaic=0.0,
    mixup=0.0,

    degrees=0.0,
    translate=0.0,
    shear=0.0,
    perspective=0.0,

    scale=0.10,

    fliplr=0.0,
    flipud=0.0,

    hsv_h=0.0,
    hsv_s=0.08,
    hsv_v=0.15,

    patience=8,

    project=str(OUTPUT_ROOT),
    name="run",
    exist_ok=True,

    save=True,
    plots=True,
    verbose=True
)




STARTING FINE-TUNING
New https://pypi.org/project/ultralytics/8.4.165 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.253 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/company_pseudo_finetune/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.08, hsv_v=0.15, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/notebooks/tanvirmahtabtaneem/image-an

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


       1/30         5G     0.7429     0.3253      1.087          8        960: 100% ━━━━━━━━━━━━ 1013/1013 2.8it/s 6:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 1.5s/it 50.8s
                   all        266       2230      0.916      0.917      0.973      0.717

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      5.06G     0.7084     0.3054      1.062         35        960: 14% ━╸────────── 143/1013 2.9it/s 50.6s<5:02

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


       2/30      5.13G     0.7011     0.3014       1.06          9        960: 100% ━━━━━━━━━━━━ 1013/1013 2.9it/s 5:52
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 5.5it/s 6.2s
                   all        266       2230      0.912      0.926      0.977      0.715

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      5.06G     0.6751      0.287      1.043         34        960: 86% ━━━━━━━━━━── 867/1013 2.8it/s 5:00<51.6s

...

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


      10/30      5.16G     0.6129     0.2608      1.005          8        960: 100% ━━━━━━━━━━━━ 1013/1013 2.9it/s 5:50
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 5.4it/s 6.2s
                   all        266       2230      0.936      0.901      0.974      0.713

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      11/30      5.07G     0.6088     0.2595      1.002         35        960: 33% ━━━╸──────── 334/1013 2.9it/s 1:56<3:54

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


      11/30      5.13G     0.6087     0.2593      1.003          9        960: 100% ━━━━━━━━━━━━ 1013/1013 2.9it/s 5:50
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 5.5it/s 6.2s
                   all        266       2230      0.937      0.905      0.977      0.713
EarlyStopping: Training stopped early as no improvement observed in last 8 epochs. Best results observed at epoch 3, best model saved as best.pt.
To update EarlyStopping(patience=8) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

11 epochs completed in 1.110 hours.
Optimizer stripped from /kaggle/working/company_pseudo_finetune/run/weights/last.pt, 40.7MB
Optimizer stripped from /kaggle/working/company_pseudo_finetune/run/weights/best.pt, 40.7MB

Validating /kaggle/working/company_pseudo_finetune/run/weights/best.pt...
Ultralytics 8.3.253 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11m s

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  31,  32,  33,  34,  35,  36,  37,  38,  39,  41,  42,  43,  44,  45,  46,  47,  48,  49,  51,  52,  53,  54,  55,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  73,  74,  75,  76,  77,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a44d4354e60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012

In [12]:
# ============================================================
# 11. LOAD NEW BEST MODEL
# ============================================================

trained_path = (
    OUTPUT_ROOT
    / "run"
    / "weights"
    / "best.pt"
)

if not trained_path.exists():
    raise FileNotFoundError(
        f"New best.pt was not created:\n"
        f"{trained_path}"
    )

shutil.copy2(
    trained_path,
    NEW_MODEL
)

new_model = YOLO(
    str(NEW_MODEL)
)

print(
    "\nNew model saved:",
    NEW_MODEL
)




New model saved: /kaggle/working/company_pseudo_finetune/best.pt


In [13]:
# ============================================================
# 12. VALIDATE ON REAL LPDB-A VALIDATION DATA
# ============================================================

print("\n" + "=" * 70)
print("LPDB-A VALIDATION")
print("=" * 70)

metrics = new_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=2
)

print(
    "Precision:",
    metrics.box.mp
)

print(
    "Recall:",
    metrics.box.mr
)

print(
    "mAP50:",
    metrics.box.map50
)

print(
    "mAP50-95:",
    metrics.box.map
)




LPDB-A VALIDATION
Ultralytics 8.3.253 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11m summary (fused): 125 layers, 20,108,674 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1980.7±836.6 MB/s, size: 78.2 KB)
val: Scanning /kaggle/working/company_pseudo_finetune/dataset/labels/val.cache... 267 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 267/267 112.0Mit/s 0.0s
val: /kaggle/working/company_pseudo_finetune/dataset/images/val/2570.jpg: ignoring corrupt image/label: Label class 103 exceeds dataset class count 102. Possible class labels are 0-101
val: /kaggle/working/company_pseudo_finetune/dataset/images/val/331.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 5.1it/s 13.1s
                   all        266       2230      0.906      0.938      0.977      0.718
                     0        123        156      0.962      0.9

In [14]:
# ============================================================
# 13. TOKEN ORDER / SEQUENCE EVALUATION
# ============================================================

def ordered_tokens_from_label(
    label_path
):

    tokens = []

    for line in label_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines():

        parts = line.split()

        if len(parts) != 5:
            continue

        try:

            class_id = int(
                float(parts[0])
            )

            cx = float(
                parts[1]
            )

            cy = float(
                parts[2]
            )

        except ValueError:

            continue

        if not (
            0 <= class_id
            < len(class_names)
        ):
            continue

        tokens.append(
            {
                "name":
                    class_names[class_id],

                "cx":
                    cx,

                "cy":
                    cy
            }
        )

    if not tokens:
        return []

    tokens.sort(
        key=lambda x:
        x["cy"]
    )

    rows = []

    for token in tokens:

        placed = False

        for row in rows:

            row_y = np.mean(
                [
                    x["cy"]
                    for x in row
                ]
            )

            if abs(
                token["cy"] - row_y
            ) <= 0.08:

                row.append(token)
                placed = True
                break

        if not placed:
            rows.append(
                [token]
            )

    rows.sort(
        key=lambda row:
        np.mean(
            [
                x["cy"]
                for x in row
            ]
        )
    )

    for row in rows:

        row.sort(
            key=lambda x:
            x["cx"]
        )

    ordered = []

    for row in rows:
        ordered.extend(row)

    return [
        x["name"]
        for x in ordered
    ]


def ordered_tokens_from_prediction(
    model,
    image
):

    result = model.predict(
        source=image,
        imgsz=IMG_SIZE,
        conf=0.25,
        device=DEVICE,
        verbose=False
    )[0]

    if result.boxes is None:
        return []

    tokens = []

    for i in range(
        len(result.boxes)
    ):

        class_id = int(
            result.boxes.cls[i].item()
        )

        if not (
            0 <= class_id
            < len(class_names)
        ):
            continue

        x1, y1, x2, y2 = (
            result.boxes.xyxy[i]
            .cpu()
            .numpy()
            .tolist()
        )

        tokens.append(
            {
                "name":
                    class_names[class_id],

                "cx":
                    (x1 + x2) / 2.0,

                "cy":
                    (y1 + y2) / 2.0
            }
        )

    if not tokens:
        return []

    tokens.sort(
        key=lambda x:
        x["cy"]
    )

    rows = []

    for token in tokens:

        placed = False

        for row in rows:

            row_y = np.mean(
                [
                    x["cy"]
                    for x in row
                ]
            )

            if abs(
                token["cy"] - row_y
            ) <= IMG_SIZE * 0.08:

                row.append(token)
                placed = True
                break

        if not placed:
            rows.append(
                [token]
            )

    rows.sort(
        key=lambda row:
        np.mean(
            [
                x["cy"]
                for x in row
            ]
        )
    )

    for row in rows:

        row.sort(
            key=lambda x:
            x["cx"]
        )

    ordered = []

    for row in rows:
        ordered.extend(row)

    return [
        x["name"]
        for x in ordered
    ]


def edit_distance(
    a,
    b
):

    previous = list(
        range(
            len(b) + 1
        )
    )

    for i, ca in enumerate(
        a,
        start=1
    ):

        current = [i]

        for j, cb in enumerate(
            b,
            start=1
        ):

            cost = (
                0
                if ca == cb
                else 1
            )

            current.append(
                min(
                    current[-1] + 1,
                    previous[j] + 1,
                    previous[j - 1] + cost
                )
            )

        previous = current

    return previous[-1]


def evaluate_model(
    model,
    pairs
):

    exact = 0
    token_sum = 0.0
    edit_sum = 0.0
    count = 0

    for i, (
        image_path,
        label_path
    ) in enumerate(
        pairs
    ):

        image = cv2.imread(
            str(image_path)
        )

        if image is None:
            continue

        gt = ordered_tokens_from_label(
            label_path
        )

        pred = ordered_tokens_from_prediction(
            model,
            image
        )

        distance = edit_distance(
            gt,
            pred
        )

        max_len = max(
            len(gt),
            len(pred),
            1
        )

        accuracy = max(
            0.0,
            1.0
            - distance / max_len
        )

        if gt == pred:
            exact += 1

        token_sum += accuracy
        edit_sum += distance
        count += 1

        if (
            (i + 1) % 50
            == 0
        ):

            print(
                f"{i + 1}/"
                f"{len(pairs)}"
            )

    return {
        "exact":
            100.0 * exact / count,

        "token":
            100.0 * token_sum / count,

        "edit":
            edit_sum / count
    }



In [15]:
# ============================================================
# 14. CURRENT vs NEW ON SAME 268 TEST IMAGES
# ============================================================

print("\n" + "=" * 70)
print("HELD-OUT LPDB-A TEST")
print("=" * 70)

current_model = YOLO(
    str(CURRENT_MODEL)
)

print(
    "\nCurrent model..."
)

current_results = evaluate_model(
    current_model,
    public_test
)

print(
    "\nNew model..."
)

new_results = evaluate_model(
    new_model,
    public_test
)

print("\nRESULTS")

print(
    "Current exact sequence:",
    f"{current_results['exact']:.2f}%"
)

print(
    "New exact sequence    :",
    f"{new_results['exact']:.2f}%"
)

print(
    "Improvement            :",
    f"{new_results['exact'] - current_results['exact']:+.2f} pp"
)

print(
    "\nCurrent token accuracy:",
    f"{current_results['token']:.2f}%"
)

print(
    "New token accuracy    :",
    f"{new_results['token']:.2f}%"
)

print(
    "Current edit distance :",
    f"{current_results['edit']:.3f}"
)

print(
    "New edit distance     :",
    f"{new_results['edit']:.3f}"
)


HELD-OUT LPDB-A TEST

Current model...
50/267
100/267
150/267
200/267
250/267

New model...
50/267
100/267
150/267
200/267
250/267

RESULTS
Current exact sequence: 56.93%
New exact sequence    : 56.93%
Improvement            : +0.00 pp

Current token accuracy: 77.94%
New token accuracy    : 77.72%
Current edit distance : 1.925
New edit distance     : 1.936


In [16]:
# ============================================================
# 15. SAVE COMPARISON CSV
# ============================================================

comparison = pd.DataFrame(
    [
        {
            "model":
                "Current",

            "exact_sequence_accuracy":
                current_results["exact"],

            "token_accuracy":
                current_results["token"],

            "mean_edit_distance":
                current_results["edit"]
        },

        {
            "model":
                "New",

            "exact_sequence_accuracy":
                new_results["exact"],

            "token_accuracy":
                new_results["token"],

            "mean_edit_distance":
                new_results["edit"]
        }
    ]
)

comparison.to_csv(
    RESULT_CSV,
    index=False
)


In [17]:
# ============================================================
# 16. ZIP THE NEW MODEL
# ============================================================

if MODEL_ZIP.exists():
    MODEL_ZIP.unlink()

with zipfile.ZipFile(
    MODEL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    z.write(
        NEW_MODEL,
        arcname="best.pt"
    )


In [18]:
# ============================================================
# 17. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("COMPLETE")
print("=" * 70)

print(
    "New best.pt:"
)

print(
    NEW_MODEL
)

print(
    "\nDownload ZIP:"
)

print(
    MODEL_ZIP
)

print(
    "\nComparison CSV:"
)

print(
    RESULT_CSV
)



COMPLETE
New best.pt:
/kaggle/working/company_pseudo_finetune/best.pt

Download ZIP:
/kaggle/working/company_bangla_best.zip

Comparison CSV:
/kaggle/working/company_pseudo_finetune/model_comparison.csv
